In [1]:
import os

import math
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, utils

from tqdm import tqdm
from PIL import Image
from glob import glob
import importlib

import ldpm_model as ldpm
import ddpm_model as ddpm
ContextUnet = ddpm.ContextUnet
VAE = ldpm.VAE

In [ ]:
class Config():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    n_feat = 128
    n_cfeat = 5
    height = 16

    timesteps = 500
    start = 1e-4
    end = 2e-2
    betas = torch.linspace(start, end, timesteps).to(device=device)
    alphas = 1.0 - betas
    alpha_bars = torch.cumprod(alphas, dim=0)

    batch_size = 256
    n_epoch = 7000
    vae_n_epoch = 2000
    lr = 1e-4

    image_size=64
    latent_dim = 128
    channels=3

    save_dir_weight = "ldpm_weight"
    save_dir_img = "ldpm_img"
    save_dir_img_vae = "vae_img"
    data_path = "datas/tinyhero"

    os.makedirs(save_dir_weight, exist_ok=True)
    os.makedirs(save_dir_img, exist_ok=True)
    os.makedirs(save_dir_img_vae, exist_ok=True)

config = Config()

/tmp/ipykernel_211740/3251417346.py:11: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  alphas = torch.tensor(1.0 - betas, device=device)


In [ ]:
def q_sample(x0, t, noise):
    sqrt_ab = torch.sqrt(config.alpha_bars[t])[:, None, None, None]
    sqrt_mab = torch.sqrt(1.0 - config.alpha_bars[t])[:, None, None, None]
    return sqrt_ab * x0 + sqrt_mab * noise

@torch.no_grad()
def sample(model, vae, n=16, latent_mean=None, latent_std=None):
    model.eval()
    vae.eval()
    
    z = torch.randn(n, config.latent_dim, config.height, config.height, device=config.device)

    if latent_mean is None:
        latent_mean = torch.zeros(1, config.latent_dim, 1, 1, device=config.device)
    if latent_std is None:
        latent_std = torch.ones(1, config.latent_dim, 1, 1, device=config.device)
    latent_mean = latent_mean.to(config.device)
    latent_std = latent_std.to(config.device)

    for t in reversed(range(config.timesteps)):
        t_batch = torch.full((n, ), t, device=config.device, dtype=torch.long)
        t_input = (t_batch.float() / (config.timesteps - 1)).unsqueeze(-1)
        
        noise_pred = model(z, t_input)

        beta_t = config.betas[t]
        alpha_t = config.alphas[t]
        alpha_bar_t = config.alpha_bars[t]

        if t > 0:
            alpha_bar_prev = config.alpha_bars[t - 1]
            beta_tilde = beta_t * (1.0 - alpha_bar_prev) / (1.0 - alpha_bar_t)
            noise = torch.randn_like(z)
        else:
            beta_tilde = torch.tensor(0.0, device=config.device)
            noise = torch.zeros_like(z)

        z = (
            (z - ((1.0 - alpha_t) / torch.sqrt(1.0 - alpha_bar_t)) * noise_pred) / torch.sqrt(alpha_t)
            + torch.sqrt(beta_tilde) * noise
        )

    z = z * latent_std + latent_mean
    out_imgs = vae.decode(z)
    out_imgs = (out_imgs.clamp(-1, 1) + 1) / 2
    return out_imgs

@torch.no_grad()
def vae_sample(vae, loader, n=16):
    vae.eval()

    x = next(iter(loader))[:n].to(config.device)
    x_recon, _ = vae(x)

    x = (x.clamp(-1, 1) + 1) / 2
    x_recon = (x_recon.clamp(-1, 1) + 1) / 2

    comparison = torch.cat([x, x_recon], dim=0)
    return comparison

In [4]:
class HeroDataset(Dataset):
    def __init__(self, root, transform=None):
        self.paths = sorted(glob(os.path.join(root, "*/*.png")))
        self.transform= transform
    
    def __len__(self):
        return len(self.paths)
    
    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img

transform = transforms.Compose([
    transforms.Resize((config.image_size, config.image_size)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

dataset = HeroDataset(config.data_path, transform)
loader = DataLoader(dataset, batch_size=config.batch_size, shuffle=True,
                    num_workers=4, pin_memory=True)

In [ ]:
def train_vae():
    vae = VAE(config.channels, config.latent_dim, config.channels, config.image_size)
    vae.to(config.device)
    optimizer_vae = torch.optim.Adam(vae.parameters(), lr=config.lr)

    for epoch in range(1, config.vae_n_epoch + 1):
        pbar = tqdm(loader, desc=f"Epoch {epoch} / {config.vae_n_epoch}")

        beta = min(1.0, epoch / max(1, config.vae_n_epoch // 2))

        for x in pbar:
            x = x.to(config.device)

            x_recon, z, mu, logvar = vae(x, return_stats=True)

            loss_recon = F.mse_loss(x_recon, x)
            loss_kl = vae.kl_loss(mu, logvar)
            loss = loss_recon + beta * loss_kl

            optimizer_vae.zero_grad()
            loss.backward()
            optimizer_vae.step()

            pbar.set_postfix({
                "loss": f"{loss.item():.4f}",
                "recon": f"{loss_recon.item():.4f}",
                "kl": f"{loss_kl.item():.4f}",
                "beta": f"{beta:.3f}",
            })

        
        if epoch % 60 == 0:
            imgs = vae_sample(vae, loader, n=16)
            utils.save_image(imgs, f"{config.save_dir_img_vae}/vae_epoch_{epoch:03d}.png", nrow=4)
            torch.save(vae.state_dict(), f"{config.save_dir_weight}/vae_{epoch}epoch.pth")
    
    torch.save(vae.state_dict(), f"{config.save_dir_weight}/vae.pth")
    print("Training Complete")

def train_latent_diffusion(vae_path=None):
    model = ContextUnet(config.latent_dim, config.n_feat, config.n_cfeat, config.height)
    model.to(config.device)

    vae = VAE(config.channels, config.latent_dim, config.channels, config.image_size)
    vae.to(config.device)
    vae.load_state_dict(torch.load(vae_path))
    vae.eval()

    optimizer_unet = torch.optim.Adam(model.parameters(), lr=config.lr)

    @torch.no_grad()
    def estimate_latent_stats(max_batches=20):
        c = config.latent_dim
        latent_sum = torch.zeros(c, device=config.device)
        latent_sq_sum = torch.zeros(c, device=config.device)
        count = 0

        for i, x in enumerate(loader):
            if i >= max_batches:
                break
            x = x.to(config.device)
            z = vae.encode(x)
            z_flat = z.permute(1, 0, 2, 3).reshape(c, -1)
            latent_sum += z_flat.sum(dim=1)
            latent_sq_sum += (z_flat ** 2).sum(dim=1)
            count += z_flat.shape[1]

        mean = latent_sum / count
        var = latent_sq_sum / count - mean ** 2
        std = torch.sqrt(torch.clamp(var, min=1e-6))

        mean = mean.view(1, c, 1, 1)
        std = std.view(1, c, 1, 1)
        return mean, std

    latent_mean, latent_std = estimate_latent_stats(max_batches=20)

    for epoch in range(1, config.n_epoch + 1):
        model.train()
        pbar = tqdm(loader, desc=f"Epoch {epoch} / {config.n_epoch}")
        for x in pbar:
            x = x.to(config.device)

            with torch.no_grad():
                z = vae.encode(x)
                z = (z - latent_mean) / latent_std

            bs = x.size(0)
            t = torch.randint(0, config.timesteps, (bs, ), device=config.device).long()
            noise = torch.randn_like(z)

            z_t = q_sample(z, t, noise)
            t_input = (t.float() / (config.timesteps - 1)).unsqueeze(-1)
            noise_pred = model(z_t, t_input)

            loss_diff = F.mse_loss(noise_pred, noise)

            optimizer_unet.zero_grad()
            loss_diff.backward()
            optimizer_unet.step()

            pbar.set_postfix({"Loss": loss_diff.item()})
        
    
        if epoch % 100 == 0:
            imgs = sample(model, vae, n=16, latent_mean=latent_mean, latent_std=latent_std)
            utils.save_image(imgs, f"{config.save_dir_img}/ldm_epoch_{epoch:03d}.png", nrow=4)
            torch.save(model.state_dict(), f"{config.save_dir_weight}/ldm_{epoch}epoch.pth")
    
    torch.save(model.state_dict(), f"{config.save_dir_weight}/ldm.pth")
    print("Training Complete")

In [6]:
train_vae()

Epoch 99 / 100: 100%|██████████| 15/15 [00:00<00:00, 23.83it/s, Loss=0.00518]


Training Complete


In [11]:
train_latent_diffusion(vae_path="/media/kim/hdd1tb/master_study/ai_architecture/ddpm/ldpm_weight/vae.pth")

Epoch 613 / 3700:   7%|▋         | 1/15 [00:00<00:03,  3.95it/s, Loss=0.0558]


KeyboardInterrupt: 